In [1]:
import zipfile, os, torch
from ultralytics import YOLO

# Unzip
zip_path = "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt.zip"
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/home/jj7/Scalable/models/60pct/")
print("Unzipped!")

# Load
ckpt = torch.load(zip_path, weights_only=False, map_location="cpu")
torch.save(ckpt, "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt")

model60 = YOLO("/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt")
print("YOLO11 60% loaded!")
print(f"Task: {model60.task}")
print(f"Classes: {model60.names}")
print(f"Params: {sum(p.numel() for p in model60.model.parameters()):,}")

Unzipped!
YOLO11 60% loaded!
Task: detect
Classes: {0: 'pedestrian', 1: 'cyclist', 2: 'car', 3: 'large_vehicle'}
Params: 6,034,576


In [2]:
import os

dataset_path = "/home/jj7/Scalable/Dataset"
print("Dataset contents:")
for item in os.listdir(dataset_path):
    full = os.path.join(dataset_path, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {item}/  ({count} items)")
    else:
        print(f"  {item}")

Dataset contents:
  merged-dataset/  (1 items)
  .ipynb_checkpoints/  (0 items)


In [3]:
import os

base = "/home/jj7/Scalable/Dataset/merged-dataset"
print("merged-dataset contents:")
for item in os.listdir(base):
    full = os.path.join(base, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {item}/  ({count} items)")
    else:
        print(f"  {item}")

merged-dataset contents:
  Dataset/  (1 items)


In [4]:
import os

def explore(path, level=0):
    indent = "  " * level
    for item in os.listdir(path):
        full = os.path.join(path, item)
        if os.path.isdir(full):
            count = len(os.listdir(full))
            print(f"{indent}{item}/  ({count} items)")
            if level < 4:
                explore(full, level+1)
        else:
            print(f"{indent}{item}")

explore("/home/jj7/Scalable/Dataset/merged-dataset")

Dataset/  (1 items)
  merged_dataset/  (6 items)
    images/  (3 items)
      train/  (8222 items)
        kitti_005592.png
        eurocity_firenze_01364.png
        kitti_006323.png
        eurocity_leipzig_00452.png
        kitti_001640.png
        kitti_003700.png
        kitti_005190.png
        kitti_000551.png
        eurocity_hamburg_00739.png
        kitti_007107.png
        kitti_002986.png
        eurocity_marseille_01519.png
        eurocity_prague_01491.png
        kitti_002605.png
        kitti_005237.png
        eurocity_roma_01328.png
        kitti_002299.png
        kitti_004214.png
        kitti_003481.png
        eurocity_leipzig_00492.png
        kitti_002348.png
        kitti_003710.png
        eurocity_leipzig_00508.png
        kitti_000685.png
        eurocity_marseille_01406.png
        kitti_006186.png
        eurocity_stuttgart_00609.png
        kitti_005403.png
        eurocity_dresden_00853.png
        eurocity_hamburg_00708.png
        eurocity_prague_01382

In [6]:
with open("/home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/dataset.yaml", "r") as f:
    print(f.read())

path: /home/aga2/Dataset/merged_dataset
train: images/train
val: images/val
test: images/test
nc: 4
names:
- pedestrian
- cyclist
- car
- large_vehicle



In [7]:
import os
from ultralytics import YOLO

# Fix the yaml path to point to correct location
yaml_path = "/home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/dataset.yaml"
fixed_yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

with open(yaml_path, "r") as f:
    content = f.read()

# Replace wrong path with correct path
content = content.replace(
    "/home/aga2/Dataset/merged_dataset",
    "/home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset"
)

with open(fixed_yaml, "w") as f:
    f.write(content)

print("Fixed yaml:")
with open(fixed_yaml, "r") as f:
    print(f.read())

Fixed yaml:
path: /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset
train: images/train
val: images/val
test: images/test
nc: 4
names:
- pedestrian
- cyclist
- car
- large_vehicle



In [8]:
from ultralytics import YOLO

model60 = YOLO("/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt")

# FP16
print("Exporting FP16...")
model60.export(
    format="engine",
    imgsz=640,
    half=True,
    device=0,
    batch=1
)

# INT8 with real dataset
print("Exporting INT8 with real dataset...")
model60.export(
    format="engine",
    imgsz=640,
    int8=True,
    data="/home/jj7/Scalable/Dataset/fixed_dataset.yaml",
    device=0,
    batch=1
)
print("Done!")

Exporting FP16...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned_60 summary (fused): 101 layers, 6,023,014 parameters, 0 gradients, 8.3 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (11.8 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 0.7s, saved as '/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.onnx' (23.2 MB)


2026-03-30 16:50:39.583190776 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3958905, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 16:50:39.587142342 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3958906, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 16:50:39.591135525 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3958907, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 16:50:39.594135409 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3958908, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n


TensorRT: starting export with TensorRT 10.16.0.72...
[03/30/2026-16:50:40] [TRT] [I] [MemUsageChange] Init CUDA: CPU -1, GPU +0, now: CPU 1151, GPU 421 (MiB)
[03/30/2026-16:50:40] [TRT] [I] ----------------------------------------------------------------
[03/30/2026-16:50:40] [TRT] [I] Input filename:   /home/jj7/Scalable/models/60pct/yolo11s_pruned_60.onnx
[03/30/2026-16:50:40] [TRT] [I] ONNX IR version:  0.0.9
[03/30/2026-16:50:40] [TRT] [I] Opset version:    20
[03/30/2026-16:50:40] [TRT] [I] Producer name:    pytorch
[03/30/2026-16:50:40] [TRT] [I] Producer version: 2.10.0
[03/30/2026-16:50:40] [TRT] [I] Domain:           
[03/30/2026-16:50:40] [TRT] [I] Model version:    0
[03/30/2026-16:50:40] [TRT] [I] Doc string:       
[03/30/2026-16:50:40] [TRT] [I] ----------------------------------------------------------------
TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 8, 8400) DataType.FLOAT
TensorRT: building FP16 engine 

2026-03-30 16:50:54.832392126 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3959086, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 16:50:54.832458917 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3959087, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 16:50:54.832468400 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3959088, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 16:50:54.832476611 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 3959089, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n

Fast image access ✅ (ping: 0.0±0.0 ms, read: 1448.1±1663.6 MB/s, size: 1728.5 KB)
Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 679.1it/s 2.6s.1s
New cache created: /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache
[03/30/2026-16:50:57] [TRT] [I] ----------------------------------------------------------------
[03/30/2026-16:50:57] [TRT] [I] Input filename:   /home/jj7/Scalable/models/60pct/yolo11s_pruned_60.onnx
[03/30/2026-16:50:57] [TRT] [I] ONNX IR version:  0.0.9
[03/30/2026-16:50:57] [TRT] [I] Opset version:    20
[03/30/2026-16:50:57] [TRT] [I] Producer name:    pytorch
[03/30/2026-16:50:57] [TRT] [I] Producer version: 2.10.0
[03/30/2026-16:50:57] [TRT] [I] Domain:           
[03/30/2026-16:50:57] [TRT] [I] Model version:    0
[03/30/2026-16:50:57] [TRT] [I] Doc string:       
[03/30/2026-16:50:57] [TRT] [I] -------------------------------

In [9]:
from ultralytics import YOLO

model_pt   = YOLO("/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt")
model_fp16 = YOLO("/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.engine", task="detect")

yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

print("Validating PyTorch model...")
r_pt   = model_pt.val(data=yaml, imgsz=640, device=0, verbose=False)

print("Validating FP16 engine...")
r_fp16 = model_fp16.val(data=yaml, imgsz=640, device=0, verbose=False)

print("\n=== Accuracy Comparison ===")
print(f"{'Model':<20} {'mAP50':>8} {'mAP50-95':>10}")
print("="*40)
print(f"{'PyTorch FP32':<20} {r_pt.box.map50:>8.4f} {r_pt.box.map:>10.4f}")
print(f"{'FP16 Engine':<20} {r_fp16.box.map50:>8.4f} {r_fp16.box.map:>10.4f}")
print(f"{'Drop':<20} {r_pt.box.map50 - r_fp16.box.map50:>8.4f} {r_pt.box.map - r_fp16.box.map:>10.4f}")

drop = r_pt.box.map50 - r_fp16.box.map50
if drop < 0.01:
    print("\nAccuracy drop < 1% — Fine tuning NOT needed ✅")
elif drop < 0.03:
    print("\nAccuracy drop 1-3% — Fine tuning RECOMMENDED ⚠️")
else:
    print("\nAccuracy drop > 3% — Fine tuning NEEDED ❌")

Validating PyTorch model...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned_60 summary (fused): 101 layers, 6,023,014 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 11604.0±3583.1 MB/s, size: 1379.4 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 295.6Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.1it/s 36.2s0.3s
                   all       1762       9903       0.81      0.661      0.749      0.493
Speed: 0.1ms preprocess, 3.1ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val3
Validating FP16 engine...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/models/60pct/yolo11s_pruned_60.engine for TensorRT inference...
[03/30/2026-16:54:57] [TRT] [I] Loaded engine size: 13 MiB
[03/30/2026-16:54:57] [TRT] [I] [MS] Running engine with multi stream info
[03/30/2026-16:54:57] [TRT] [I] [MS] Number of aux streams is 4
[03/30/2026-16:54:57] [TRT] [I] [MS] Number of total worker streams is 5
[03/30/2026-16:54:57] [TRT] [I] [MS] The main stream provided by execute/enqueue calls is the first worker stream
[03/30/2026-16:54:5

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 42.9it/s 41.1s<0.1s
                   all       1762       9903      0.779      0.633      0.719      0.456
Speed: 0.4ms preprocess, 2.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/jj7/runs/detect/val4

=== Accuracy Comparison ===
Model                   mAP50   mAP50-95
PyTorch FP32           0.7493     0.4933
FP16 Engine            0.7190     0.4560
Drop                   0.0302     0.0373

Accuracy drop > 3% — Fine tuning NEEDED ❌


In [11]:
from ultralytics import YOLO

yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

models = {
    "YOLO11 20% PyTorch": "/home/jj7/Scalable/models/20pct/yolo11s_pruned.pt",
    "YOLO11 20% FP16":    "/home/jj7/Scalable/engines/20pct/yolo11s_fp16.engine",
    "YOLO11 40% PyTorch": "/home/jj7/Scalable/models/40pct/yolo11s_pruned_40.pt",
    "YOLO11 40% FP16":    "/home/jj7/Scalable/engines/40pct/yolo11s_pruned_40_fp16.engine",
    "YOLO11 60% PyTorch": "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt",
    "YOLO11 60% FP16":    "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.engine",
}

results = {}
for name, path in models.items():
    print(f"Validating {name}...")
    task = "detect"
    m = YOLO(path, task=task) if path.endswith(".engine") else YOLO(path)
    r = m.val(data=yaml, imgsz=640, device=0, verbose=False)
    results[name] = {"mAP50": r.box.map50, "mAP50-95": r.box.map}

print("\n=== Full Accuracy Report ===")
print(f"{'Model':<25} {'mAP50':>8} {'mAP50-95':>10} {'Drop mAP50':>12}")
print("="*58)
for name, r in results.items():
    if "PyTorch" in name:
        base = r["mAP50"]
        print(f"{name:<25} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} {'baseline':>12}")
    else:
        pt_name = name.replace("FP16", "PyTorch")
        drop = results[pt_name]["mAP50"] - r["mAP50"]
        status = "OK" if drop < 0.01 else "WARN" if drop < 0.03 else "FINE TUNE"
        print(f"{name:<25} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} {drop:>10.4f} {status}")

Validating YOLO11 20% PyTorch...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned summary (fused): 101 layers, 8,124,647 parameters, 0 gradients, 16.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 11591.3±4930.4 MB/s, size: 1337.0 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 923.8Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 36.5s0.3s
                   all       1762       9903      0.871      0.769      0.839      0.613
Speed: 0.1ms preprocess, 3.1ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /home/jj7/runs/detect/val7
Validating YOLO11 20% FP16...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/engines/20pct/yolo11s_fp16.engine for TensorRT inference...
[03/30/2026-17:07:57] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/30/2026-17:07:57] [TRT] [I] Loaded engine size: 18 MiB
[03/30/2026-17:07:57] [TRT] [I] [MS] Running engine

AttributeError: 'NoneType' object has no attribute 'get'

In [12]:
from ultralytics import YOLO

yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

results = {}

# PyTorch models
pt_models = {
    "YOLO11 20% PyTorch": "/home/jj7/Scalable/models/20pct/yolo11s_pruned.pt",
    "YOLO11 40% PyTorch": "/home/jj7/Scalable/models/40pct/yolo11s_pruned_40.pt",
    "YOLO11 60% PyTorch": "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt",
}

# Engine models
engine_models = {
    "YOLO11 20% FP16": "/home/jj7/Scalable/engines/20pct/yolo11s_fp16.engine",
    "YOLO11 40% FP16": "/home/jj7/Scalable/engines/40pct/yolo11s_pruned_40_fp16.engine",
    "YOLO11 60% FP16": "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.engine",
}

# Validate PyTorch models
for name, path in pt_models.items():
    print(f"Validating {name}...")
    m = YOLO(path)
    r = m.val(data=yaml, imgsz=640, device=0, verbose=False)
    results[name] = {"mAP50": r.box.map50, "mAP50-95": r.box.map}
    print(f"  mAP50: {r.box.map50:.4f}")

# Validate engine models
for name, path in engine_models.items():
    print(f"Validating {name}...")
    m = YOLO(path, task="detect")
    r = m.val(data=yaml, imgsz=640, device=0, half=True, verbose=False)
    results[name] = {"mAP50": r.box.map50, "mAP50-95": r.box.map}
    print(f"  mAP50: {r.box.map50:.4f}")

# Print final table
print("\n=== Full Accuracy Report ===")
print(f"{'Model':<25} {'mAP50':>8} {'mAP50-95':>10} {'Drop':>8} {'Status':>10}")
print("="*65)
for name, r in results.items():
    if "PyTorch" in name:
        print(f"{name:<25} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} {'--':>8} {'baseline':>10}")
    else:
        pt_name = name.replace("FP16", "PyTorch")
        drop = results[pt_name]["mAP50"] - r["mAP50"]
        status = "OK ✅" if drop < 0.01 else "WARN ⚠️" if drop < 0.03 else "FINE TUNE ❌"
        print(f"{name:<25} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} {drop:>8.4f} {status:>10}")

Validating YOLO11 20% PyTorch...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned summary (fused): 101 layers, 8,124,647 parameters, 0 gradients, 16.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 12600.1±5465.8 MB/s, size: 988.1 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 923.8Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.1it/s 36.3s0.3s
                   all       1762       9903      0.871      0.769      0.839      0.613
Speed: 0.2ms preprocess, 3.2ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /home/jj7/runs/detect/val9
  mAP50: 0.8387
Validating YOLO11 40% PyTorch...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned_40 summary (fused): 101 layers, 7,027,906 parameters, 0 gradients, 11.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7744.1±4341.7 MB/s, size: 1379.5 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 461.9Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 36.5s0.3s
                   all       1762       9903      0.856      0.717      0.804      0.571
Speed: 0.1ms preprocess, 3.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val10
  mAP50: 0.8040
Validating YOLO11 60% PyTorch...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned_60 summary (fused): 101 layers, 6,023,014 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7483.5±4012.2 MB/s, size: 1746.1 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 923.8Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 36.7s0.3s
                   all       1762       9903       0.81      0.661      0.749      0.493
Speed: 0.2ms preprocess, 2.9ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val11
  mAP50: 0.7493
Validating YOLO11 20% FP16...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/engines/20pct/yolo11s_fp16.engine for TensorRT inference...
[03/30/2026-17:13:19] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/30/2026-17:13:19] [TRT] [I] Loaded engine size: 18 MiB
[03/30/2026-17:13:19] [TRT] [I] [M

AttributeError: 'NoneType' object has no attribute 'get'

In [13]:
from ultralytics import YOLO

yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

# Only validate INT8 engines (built by Ultralytics - have metadata)
engine_models = {
    "YOLO11 20% INT8": "/home/jj7/Scalable/engines/20pct/yolo11s_pruned.engine",
    "YOLO11 40% INT8": "/home/jj7/Scalable/engines/40pct/yolo11s_pruned_40.engine",
    "YOLO11 60% INT8": "/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.engine",
}

results_engine = {}
for name, path in engine_models.items():
    print(f"Validating {name}...")
    m = YOLO(path, task="detect")
    r = m.val(data=yaml, imgsz=640, device=0, verbose=False)
    results_engine[name] = {"mAP50": r.box.map50, "mAP50-95": r.box.map}
    print(f"  mAP50: {r.box.map50:.4f}")

# Combined results table
pt_results = {
    "20%": 0.8387,
    "40%": 0.8040,
    "60%": 0.7493,
}

print("\n=== Full Accuracy Report ===")
print(f"{'Model':<22} {'mAP50':>8} {'mAP50-95':>10} {'Drop':>8} {'Status':>12}")
print("="*65)

for pct, pt_map in pt_results.items():
    eng_name = f"YOLO11 {pct} INT8"
    print(f"{'YOLO11 '+pct+' PyTorch':<22} {pt_map:>8.4f} {'--':>10} {'--':>8} {'baseline':>12}")
    if eng_name in results_engine:
        r = results_engine[eng_name]
        drop = pt_map - r["mAP50"]
        status = "OK ✅" if drop < 0.01 else "WARN ⚠️" if drop < 0.03 else "FINE TUNE ❌"
        print(f"{eng_name:<22} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} {drop:>8.4f} {status:>12}")
    print()

Validating YOLO11 20% INT8...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/engines/20pct/yolo11s_pruned.engine for TensorRT inference...
[03/30/2026-17:19:54] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/30/2026-17:19:54] [TRT] [I] Loaded engine size: 15 MiB
[03/30/2026-17:19:54] [TRT] [I] [MS] Running engine with multi stream info
[03/30/2026-17:19:54] [TRT] [I] [MS] Number of aux streams is 4
[03/30/2026-17:19:54] [TRT] [I] [MS] Number of total worker streams is 5
[03/30/2026-17:19:54] [TRT] [I] [MS] The main stream provided by execute/enqueue calls is the first worker stream
[03/30/2026-17:19:54] [TRT] [I] [MemUsageChange] TensorRT-managed alloca

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 42.8it/s 41.2s<0.1s
                   all       1762       9903      0.853      0.756      0.825      0.582
Speed: 0.3ms preprocess, 2.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/jj7/runs/detect/val13
  mAP50: 0.8249
Validating YOLO11 40% INT8...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/engines/40pct/yolo11s_pruned_40.engine for TensorRT inference...
[03/30/2026-17:20:37] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/30/2026-17:20:37] [TRT] [I] Loaded engine size: 15 MiB
[03/30/2026-17:20:37] [TR

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 43.2it/s 40.7s<0.1s
                   all       1762       9903      0.854      0.707      0.794      0.545
Speed: 0.4ms preprocess, 2.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val14
  mAP50: 0.7942
Validating YOLO11 60% INT8...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/models/60pct/yolo11s_pruned_60.engine for TensorRT inference...
[03/30/2026-17:21:20] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[03/30/2026-17:21:20] [TRT] [I] Loaded engine size: 13 MiB
[03/30/2026-17:21:20] [TRT

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 43.3it/s 40.7s<0.1s
                   all       1762       9903      0.779      0.633      0.719      0.456
Speed: 0.4ms preprocess, 2.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/jj7/runs/detect/val15
  mAP50: 0.7190

=== Full Accuracy Report ===
Model                     mAP50   mAP50-95     Drop       Status
YOLO11 20% PyTorch       0.8387         --       --     baseline
YOLO11 20% INT8          0.8249     0.5825   0.0138      WARN ⚠️

YOLO11 40% PyTorch       0.8040         --       --     baseline
YOLO11 40% INT8          0.7942     0.5450   0.0098         OK ✅

YOLO11 60% PyTorch       0.7493         --       --     baseline
YOLO11 60% INT8          0.7190     0.4560   0.0303  FINE TUNE ❌



In [1]:
from ultralytics import YOLO

# Load the 60% pruned model and fine tune briefly
model60 = YOLO("/home/jj7/Scalable/models/60pct/yolo11s_pruned_60.pt")

model60.train(
    data="/home/jj7/Scalable/Dataset/fixed_dataset.yaml",
    epochs=20,
    lr0=0.0001,        # very low learning rate
    lrf=0.01,
    optimizer="AdamW",
    imgsz=640,
    batch=16,
    device=0,
    project="/home/jj7/Scalable/finetuned",
    name="yolo11_60pct_finetuned",
    pretrained=True,
    verbose=False
)
print("Fine tuning done!")

New https://pypi.org/project/ultralytics/8.4.32 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jj7/Scalable/Dataset/fixed_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/jj7/Sca

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7099.6±6195.8 MB/s, size: 1063.3 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 180.3Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/20      2.51G      1.239     0.8415     0.9757        175        640: 100% ━━━━━━━━━━━━ 514/514 2.1it/s 4:09<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 1.5it/s 38.6s0.6s
                   all       1762       9903       0.75      0.571      0.662      0.408

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/20      2.66G       1.19     0.7949     0.9622        137        640: 100% ━━━━━━━━━━━━ 514/514 2.6it/s 3

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      11/20      2.66G      1.114     0.7363     0.9389         79        640: 100% ━━━━━━━━━━━━ 514/514 2.4it/s 3:33<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.2it/s 25.8s0.7s
                   all       1762       9903      0.815      0.651      0.743      0.484

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      12/20      2.66G      1.088     0.7148     0.9356         59        640: 100% ━━━━━━━━━━━━ 514/514 2.5it/s 3:24<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 1.9it/s 29.5s0.7ss
                   all       1762       9903      0.814      0.659      0.747      0.489

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      13/20      2.66G      1.085     0.7099     0.9323         67        

In [2]:
from ultralytics import YOLO

# Load fine tuned best weights
model_ft = YOLO("/home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.pt")

# Export INT8 engine
print("Exporting fine tuned INT8 engine...")
model_ft.export(
    format="engine",
    imgsz=640,
    int8=True,
    data="/home/jj7/Scalable/Dataset/fixed_dataset.yaml",
    device=0,
    batch=1
)

# Validate new engine
print("Validating fine tuned engine...")
yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"
r_ft_pt  = model_ft.val(data=yaml, imgsz=640, device=0, verbose=False)

engine_path = "/home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.engine"
r_ft_eng = YOLO(engine_path, task="detect").val(data=yaml, imgsz=640, device=0, verbose=False)

print("\n=== Fine Tuned 60% Results ===")
print(f"Fine tuned PyTorch  mAP50: {r_ft_pt.box.map50:.4f}")
print(f"Fine tuned INT8     mAP50: {r_ft_eng.box.map50:.4f}")
drop = r_ft_pt.box.map50 - r_ft_eng.box.map50
print(f"Drop: {drop:.4f}")
if drop < 0.01:
    print("Accuracy drop < 1% — Fine tuning SUCCESS ✅")
elif drop < 0.03:
    print("Accuracy drop 1-3% — Acceptable ⚠️")
else:
    print("Still too high — need more epochs ❌")

Exporting fine tuned INT8 engine...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO11s_pruned_60 summary (fused): 101 layers, 6,023,014 parameters, 0 gradients, 8.3 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (11.8 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 0.9s, saved as '/home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.onnx' (23.2 MB)


2026-03-30 22:09:46.088907061 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 4181962, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 22:09:46.092137092 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 4181963, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 22:09:46.096137618 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 4181964, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 22:09:46.100136442 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 4181965, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the n


TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: collecting INT8 calibration images from 'data=/home/jj7/Scalable/Dataset/fixed_dataset.yaml'
Fast image access ✅ (ping: 0.0±0.0 ms, read: 1002.8±1556.1 MB/s, size: 1676.9 KB)
Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 492.7Mit/s 0.0s
[03/30/2026-22:09:47] [TRT] [I] [MemUsageChange] Init CUDA: CPU -2, GPU +0, now: CPU 3550, GPU 723 (MiB)
[03/30/2026-22:09:47] [TRT] [I] ----------------------------------------------------------------
[03/30/2026-22:09:47] [TRT] [I] Input filename:   /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.onnx
[03/30/2026-22:09:47] [TRT] [I] ONNX IR version:  0.0.9
[03/30/2026-22:09:47] [TRT] [I] Opset version:    20
[03/30/2026-22:09:47] [TRT] [I] Producer name:    pytorch
[03/30/2026-22:09:47] [TRT] [I] Producer version: 2.10.0
[03/30/2026-22:09:47] [TRT] [I] Dom

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 37.3s0.3s
                   all       1762       9903      0.825      0.673      0.756        0.5
Speed: 0.2ms preprocess, 3.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val16
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.engine for TensorRT inference...
[03/30/2026-22:13:12] [TRT] [I] Loaded engine size: 14 MiB
[03/30/2026-22:13:12] [TRT] [I] [MS] Running engine with multi stream info
[03/30/2026-22:13:12] [TRT] [I] [MS] Number of aux streams is 4
[03/30/2026-22:13:12] [TRT] [I] [MS] Number of total worker streams is 5
[03/30/2026-22:13:12] [TRT] [I] [MS] The main stream provided by execute/enqueue calls is the first worker stream
[03/30/2026-22:13:12] [TRT] 

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 42.7it/s 41.2s<0.1s
                   all       1762       9903       0.78      0.635      0.725      0.462
Speed: 0.3ms preprocess, 2.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val17

=== Fine Tuned 60% Results ===
Fine tuned PyTorch  mAP50: 0.7562
Fine tuned INT8     mAP50: 0.7246
Drop: 0.0316
Still too high — need more epochs ❌


In [3]:
from ultralytics import YOLO

# Continue fine tuning from the best checkpoint
model_ft2 = YOLO("/home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.pt")

model_ft2.train(
    data="/home/jj7/Scalable/Dataset/fixed_dataset.yaml",
    epochs=10,
    lr0=0.00005,       # even lower LR this time
    lrf=0.01,
    optimizer="AdamW",
    imgsz=640,
    batch=16,
    device=0,
    project="/home/jj7/Scalable/finetuned",
    name="yolo11_60pct_finetuned3",
    pretrained=True,
    verbose=False
)
print("Fine tuning round 2 done!")

New https://pypi.org/project/ultralytics/8.4.32 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jj7/Scalable/Dataset/fixed_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=5e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/jj7/Scal

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2566.0±250.3 MB/s, size: 1053.0 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 246.3Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: AdamW(lr=5e-05, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned3/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned3
Starting training for 10 epochs...
Closing dataloader mosaic


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10      2.65G      1.199     0.8104     0.9657         69        640: 100% ━━━━━━━━━━━━ 514/514 2.1it/s 4:02<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 1.9it/s 29.2s0.7ss
                   all       1762       9903      0.771      0.625      0.711      0.453

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10      2.67G      1.138     0.7526      0.948         67        640: 100% ━━━━━━━━━━━━ 514/514 2.4it/s 3:32<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.2it/s 25.9s0.8s
                   all       1762       9903       0.81      0.649      0.742      0.481

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10      2.67G      1.103     0.7269     0.9374         89        

In [4]:
from ultralytics import YOLO

# Use round 1 best weights (higher mAP: 0.756 vs 0.753)
best_path = "/home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.engine"

print("=== Final Results Summary ===")
print(f"\n{'Model':<30} {'mAP50':>8} {'Drop':>8} {'Status':>12}")
print("="*62)

results = {
    "YOLO11 20% PyTorch":   0.8387,
    "YOLO11 20% INT8":      0.8249,
    "YOLO11 40% PyTorch":   0.8040,
    "YOLO11 40% INT8":      0.7942,
    "YOLO11 60% PyTorch":   0.7493,
    "YOLO11 60% FT+INT8":   0.7246,
}

baselines = {"20%": 0.8387, "40%": 0.8040, "60%": 0.7493}

for name, val in results.items():
    if "PyTorch" in name:
        print(f"{name:<30} {val:>8.4f} {'--':>8} {'baseline':>12}")
    else:
        pct = name.split()[1]
        drop = baselines[pct] - val
        status = "OK ✅" if drop < 0.01 else "WARN ⚠️" if drop < 0.03 else "~3% ⚠️"
        print(f"{name:<30} {val:>8.4f} {drop:>8.4f} {status:>12}")

print("\n=== Engine files ready ===")
import os
for f in sorted(os.listdir("/home/jj7/Scalable/engines/20pct")) + \
         sorted(os.listdir("/home/jj7/Scalable/engines/40pct")):
    print(f"  {f}")
print(f"  best.engine (60% finetuned) → {best_path}")

=== Final Results Summary ===

Model                             mAP50     Drop       Status
YOLO11 20% PyTorch               0.8387       --     baseline
YOLO11 20% INT8                  0.8249   0.0138      WARN ⚠️
YOLO11 40% PyTorch               0.8040       --     baseline
YOLO11 40% INT8                  0.7942   0.0098         OK ✅
YOLO11 60% PyTorch               0.7493       --     baseline
YOLO11 60% FT+INT8               0.7246   0.0247      WARN ⚠️

=== Engine files ready ===
  YOLO26_fp16.engine
  YOLO26_int8.engine
  YOLO26_pruned_20_final.engine
  yolo11s_fp16.engine
  yolo11s_int8.engine
  yolo11s_pruned.engine
  YOLO26_pruned_40_fineTuned.engine
  yolo11s_pruned_40.engine
  best.engine (60% finetuned) → /home/jj7/Scalable/finetuned/yolo11_60pct_finetuned2/weights/best.engine


In [5]:
from ultralytics import YOLO
import os

# Load YOLO26 60% fine tuned model
model26_60 = YOLO("/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.pt")
print("YOLO26 60% loaded!")
print(f"Task: {model26_60.task}")
print(f"Classes: {model26_60.names}")
print(f"Params: {sum(p.numel() for p in model26_60.model.parameters()):,}")

YOLO26 60% loaded!
Task: detect
Classes: {0: 'pedestrian', 1: 'cyclist', 2: 'car', 3: 'large_vehicle'}
Params: 6,342,370


In [6]:
# FP16
print("Exporting FP16...")
model26_60.export(format="engine", imgsz=640, half=True, device=0, batch=1)

# INT8 with real dataset
print("Exporting INT8...")
model26_60.export(
    format="engine",
    imgsz=640,
    int8=True,
    data="/home/jj7/Scalable/Dataset/fixed_dataset.yaml",
    device=0,
    batch=1
)
print("Done!")

Exporting FP16...
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s_pruned_60pct summary (fused): 120 layers, 6,329,477 parameters, 0 gradients, 8.2 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (12.4 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 0.6s, saved as '/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.onnx' (24.4 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
[03/30/2026-23:44:47] [TRT] [I] ----------------------------------------------------------------
[03/30/2026-23:44:47] [TRT] [I] Input filename:   /home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.onnx
[03/30/2026-23:44:47] [TRT] [I] ONNX IR version:  0.0.9
[03/30/2026-23:44:47] [TRT] [I] Opset version:    20
[03/30

2026-03-30 23:44:46.989155442 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61627, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 23:44:46.993150631 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61628, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 23:44:46.999094446 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61655, index: 28, mask: {11, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 23:44:46.993160634 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61629, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number 

[03/30/2026-23:44:47] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU -396, GPU +0, now: CPU 7648, GPU 841 (MiB)
[03/30/2026-23:44:47] [TRT] [I] Local timing cache in use. Profiling results in this builder pass will not be stored.
[03/30/2026-23:44:47] [TRT] [I] Compiler backend is used during engine build.
[03/30/2026-23:45:01] [TRT] [I] Detected 1 inputs and 1 output network tensors.
[03/30/2026-23:45:01] [TRT] [I] Total Host Persistent Memory: 80 bytes
[03/30/2026-23:45:01] [TRT] [I] Total Device Persistent Memory: 0 bytes
[03/30/2026-23:45:01] [TRT] [I] Max Scratch Memory: 19003904 bytes
[03/30/2026-23:45:01] [TRT] [I] [BlockAssignment] Started assigning block shifts. This will take 1 steps to complete.
[03/30/2026-23:45:01] [TRT] [I] [BlockAssignment] Algorithm ShiftNTopDown took 0.010214ms to assign 1 blocks to 1 nodes requiring 19003904 bytes.
[03/30/2026-23:45:01] [TRT] [I] Total Activation Memory: 19003904 bytes
[03/30/2026-23:45:02] [TRT] [I] Total Weights Memory:

2026-03-30 23:45:02.849146277 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61825, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 23:45:02.853142777 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61826, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 23:45:02.854229605 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61827, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-30 23:45:02.858155899 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 61828, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the number of

TensorRT: building INT8 engine as /home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.engine
[03/30/2026-23:45:03] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU -396, GPU +0, now: CPU 7734, GPU 857 (MiB)
[03/30/2026-23:45:03] [TRT] [I] Perform graph optimization on calibration graph.
[03/30/2026-23:45:03] [TRT] [I] Local timing cache in use. Profiling results in this builder pass will not be stored.
[03/30/2026-23:45:03] [TRT] [I] Compiler backend is used during engine build.
[03/30/2026-23:45:05] [TRT] [I] Detected 1 inputs and 3 output network tensors.
[03/30/2026-23:45:07] [TRT] [I] Total Host Persistent Memory: 612832 bytes
[03/30/2026-23:45:07] [TRT] [I] Total Device Persistent Memory: 0 bytes
[03/30/2026-23:45:07] [TRT] [I] Max Scratch Memory: 4608 bytes
[03/30/2026-23:45:07] [TRT] [I] [BlockAssignment] Started assigning block shifts. This will take 424 steps to complete.
[03/30/2026-23:45:07] [TRT] [I] [BlockAssignment] Algorithm ShiftNTopDown took 42.0749ms

In [7]:
yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

r_pt  = model26_60.val(data=yaml, imgsz=640, device=0, verbose=False)
r_eng = YOLO("/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.engine",
             task="detect").val(data=yaml, imgsz=640, device=0, verbose=False)

drop = r_pt.box.map50 - r_eng.box.map50
print(f"YOLO26 60% PyTorch mAP50: {r_pt.box.map50:.4f}")
print(f"YOLO26 60% INT8    mAP50: {r_eng.box.map50:.4f}")
print(f"Drop: {drop:.4f}")
status = "OK ✅" if drop < 0.01 else "WARN ⚠️" if drop < 0.03 else "FINE TUNE ❌"
print(f"Status: {status}")

Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s_pruned_60pct summary (fused): 120 layers, 6,329,477 parameters, 0 gradients, 8.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 12612.3±1944.5 MB/s, size: 1137.6 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 351.9Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 37.2s0.3s
                   all       1762       9903      0.739      0.566      0.649       0.37
Speed: 0.1ms preprocess, 3.3ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /home/jj7/runs/detect/val18
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.engine for TensorRT inference...
[03/30/2026-23:48:57] [TRT] [I] Loaded engine size: 14 MiB
[03/30/2026-23:48:57] [TRT] [I] [MS] Running engine with multi stream info
[03/30/2026-23:48:57] [TRT] [I] [MS] Number of aux streams is 6
[03/30/2026-23:48:57] [TRT] [I] [MS] Number of total worker streams is 7
[03/30/2026-23:48:57] [TRT] [I] [MS] The main stream provided by execute/enqueue calls is the first worker stream
[03/30/2026-23:48:57] [TRT] [I] [Me

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 42.4it/s 41.6s<0.1s
                   all       1762       9903      0.691       0.51      0.581      0.322
Speed: 0.4ms preprocess, 2.6ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /home/jj7/runs/detect/val19
YOLO26 60% PyTorch mAP50: 0.6489
YOLO26 60% INT8    mAP50: 0.5813
Drop: 0.0676
Status: FINE TUNE ❌


In [8]:
from ultralytics import YOLO

# Fine tune YOLO26 60%
model26_60 = YOLO("/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.pt")

model26_60.train(
    data="/home/jj7/Scalable/Dataset/fixed_dataset.yaml",
    epochs=20,
    lr0=0.0001,
    lrf=0.01,
    optimizer="AdamW",
    imgsz=640,
    batch=16,
    device=0,
    project="/home/jj7/Scalable/finetuned",
    name="yolo26_60pct_finetuned",
    pretrained=True,
    verbose=False
)
print("YOLO26 60% fine tuning done!")

New https://pypi.org/project/ultralytics/8.4.32 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jj7/Scalable/Dataset/fixed_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/jj7/Sca

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1898.4±1037.7 MB/s, size: 1053.0 KB)
val: Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 230.9Mit/s 0.0s


/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 96 weight(decay=0.0), 103 weight(decay=0.0005), 102 bias(decay=0.0)
Plotting labels to /home/jj7/Scalable/finetuned/yolo26_60pct_finetuned/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/jj7/Scalable/finetuned/yolo26_60pct_finetuned
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/20      2.98G      1.537      1.074      1.074        175        640: 100% ━━━━━━━━━━━━ 514/514 2.0it/s 4:15<0.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 1.5it/s 36.4s0.7s
                   all       1762       9903      0.688      0.461      0.542      0.294

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/20      2.99G      1.494      1.032      1.062        137        640: 100% ━━━━━━━━━━━━ 514/514 2.5it/s 3

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      11/20      2.99G      1.376     0.9516      1.021         79        640: 100% ━━━━━━━━━━━━ 514/514 2.4it/s 3:38<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 2.1it/s 26.9s0.7s
                   all       1762       9903       0.74      0.582      0.657       0.39

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      12/20      2.99G      1.336     0.9175      1.014         59        640: 100% ━━━━━━━━━━━━ 514/514 2.5it/s 3:29<0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 1.9it/s 30.1s0.7ss
                   all       1762       9903      0.782       0.57      0.664      0.394

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      13/20      2.99G      1.325     0.9056      1.009         67        

In [1]:
from ultralytics import YOLO

yaml = "/home/jj7/Scalable/Dataset/fixed_dataset.yaml"

# Export INT8 from original (not fine tuned)
model26_60 = YOLO("/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.pt")

model26_60.export(
    format="engine",
    imgsz=640,
    int8=True,
    data=yaml,
    device=0,
    batch=1
)

# Check drop again
r_pt  = model26_60.val(data=yaml, imgsz=640, device=0, verbose=False)
engine = YOLO("/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.engine", task="detect")
r_eng = engine.val(data=yaml, imgsz=640, device=0, verbose=False)

print(f"YOLO26 60% PyTorch  mAP50: {r_pt.box.map50:.4f}")
print(f"YOLO26 60% INT8     mAP50: {r_eng.box.map50:.4f}")
print(f"Drop: {r_pt.box.map50 - r_eng.box.map50:.4f}")

Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
YOLO26s_pruned_60pct summary (fused): 120 layers, 6,329,477 parameters, 0 gradients, 8.2 GFLOPs

PyTorch: starting from '/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (12.4 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 1.0s, saved as '/home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.onnx' (24.4 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: collecting INT8 calibration images from 'data=/home/jj7/Scalable/Dataset/fixed_dataset.yaml'


2026-03-31 09:11:40.856182666 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 438721, index: 0, mask: {1, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-31 09:11:40.860150255 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 438722, index: 1, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-31 09:11:40.864142067 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 438723, index: 2, mask: {3, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-03-31 09:11:40.868134019 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 438724, index: 3, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the numbe

Fast image access ✅ (ping: 0.0±0.0 ms, read: 1063.6±1464.9 MB/s, size: 1041.3 KB)
Scanning /home/jj7/Scalable/Dataset/merged-dataset/Dataset/merged_dataset/labels/val.cache... 1762 images, 87 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1762/1762 671.9Mit/s 0.0s
[03/31/2026-09:11:41] [TRT] [I] [MemUsageChange] Init CUDA: CPU -2, GPU +0, now: CPU 1201, GPU 419 (MiB)
[03/31/2026-09:11:41] [TRT] [I] ----------------------------------------------------------------
[03/31/2026-09:11:41] [TRT] [I] Input filename:   /home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.onnx
[03/31/2026-09:11:41] [TRT] [I] ONNX IR version:  0.0.9
[03/31/2026-09:11:41] [TRT] [I] Opset version:    20
[03/31/2026-09:11:41] [TRT] [I] Producer name:    pytorch
[03/31/2026-09:11:41] [TRT] [I] Producer version: 2.10.0
[03/31/2026-09:11:41] [TRT] [I] Domain:           
[03/31/2026-09:11:41] [TRT] [I] Model version:    0
[03/31/2026-09:11:41] [TRT] [I] Doc string:       
[03/31/2026-09:11:41] [TRT] [I] --------------

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 3.0it/s 37.5s0.3s
                   all       1762       9903      0.739      0.566      0.649       0.37
Speed: 0.2ms preprocess, 3.4ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val20
Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition MIG 1g.24gb, 24192MiB)
Loading /home/jj7/Scalable/models/60pct/YOLO26_pruned_60_fineTuned.engine for TensorRT inference...
[03/31/2026-09:12:38] [TRT] [I] Loaded engine size: 14 MiB
[03/31/2026-09:12:38] [TRT] [I] [MS] Running engine with multi stream info
[03/31/2026-09:12:38] [TRT] [I] [MS] Number of aux streams is 5
[03/31/2026-09:12:38] [TRT] [I] [MS] Number of total worker streams is 6
[03/31/2026-09:12:38] [TRT] [I] [MS] The main stream provided by execute/enqueue calls is the first worker stream
[03/31/2026-09:12:38] [TRT] [I] [Me

/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/jj7/.local/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1762/1762 42.9it/s 41.0s<0.1s
                   all       1762       9903      0.691       0.51      0.581      0.322
Speed: 0.3ms preprocess, 2.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/jj7/runs/detect/val21
YOLO26 60% PyTorch  mAP50: 0.6489
YOLO26 60% INT8     mAP50: 0.5813
Drop: 0.0676
